# Lab 6.4 &mdash; Citations Bound to Spans, and Refusing

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Bind every claim to the exact characters that support it
- Drop a claim that cannot name its source &mdash; before it ships, not after
- Refuse when the corpus cannot answer, and say what is missing
- Prove that neither behaviour depends on the model choosing to co-operate

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Extractive grounding.** Every claim here is a quotation, so the binding is exact
> and a citation is checkable by string comparison. Looser generation needs the
> faithfulness score from Lab 6.5 &mdash; but this is the version you can prove.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-6-04")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the corpus (synthetic, self-contained)
# Two short operating documents. Read 3.2: the rule and the exception that qualifies it are
# adjacent sentences, which is the whole of Lab 6.1's first lesson. Note also what is NOT
# here -- there is nothing about FX or hedging anywhere, and Lab 6.4 needs that gap.

DOCS = {
    "ops-runbook-v4.md": """
## 3.1 Insufficient funds
A payment returned INSUFFICIENT_FUNDS is retried once after 24 hours. If the retry also fails,
notify the client desk. Operations must not fund the account manually.

## 3.2 Limit breaches
Payments above USD 500,000 require Treasury approval before release. This does not apply to
intra-group transfers, which settle same-day without any approval.

## 3.3 Invalid beneficiary details
A payment returned INVALID_IBAN is returned to the originator with code R04. Beneficiary
details are never repaired in-house.

## 3.4 Sanctions review
A payment held for SANCTIONS_REVIEW is decided by Compliance. Operations must not release or
cancel it under any circumstances.
""",
    "escalation-policy-v2.md": """
## 1 Approval authority
A duty manager may approve a release up to USD 250,000. Above that figure Treasury approval is
required, and must be recorded against the payment reference.

## 2 Escalation timers
If an approver has not responded within 15 minutes, escalate to the Treasury lead, and after a
further 15 minutes to the head of operations.
""",
}

print(f"{len(DOCS)} documents, {sum(len(d) for d in DOCS.values())} characters")

In [ ]:
# ------------------------------------------------- carried forward from Lab 6.1 (nothing to fill in)
import re

SECTION_RE = re.compile(r"^##\s+(.*)$", re.M)
STOP = set("""a an the of for is are was were do does did what which who this that these those it
its to in on at by with from about and or not no be been have has had can could should would will
you your we our i me my how why when where there here as if then than so such only just also very
more most some any other""".split())

def terms(text):
    return {w for w in re.findall(r"[a-z0-9_]+", (text or "").lower())
            if w not in STOP and len(w) > 1}

def chunk_by_section(source, text):
    out, parts = [], SECTION_RE.split(text)
    for i in range(1, len(parts) - 1, 2):
        heading, body = parts[i].strip(), " ".join(parts[i + 1].split())
        out.append({"source": source, "section": heading, "text": heading + " -- " + body})
    return out

INDEX = [c for source, text in DOCS.items() for c in chunk_by_section(source, text)]

def similarity(query, chunk):
    q = terms(query)
    return len(q & terms(chunk["text"])) / len(q) if q else 0.0

def search(query, index=None, k=4, floor=0.0):
    index = INDEX if index is None else index
    scored = sorted(((similarity(query, c), c) for c in index), key=lambda sc: -sc[0])
    return [{"score": round(s, 3), **c} for s, c in scored[:k] if s >= floor]

print(f"index: {len(INDEX)} chunks")

## Concept

Two behaviours a regulated client will ask about, and both have to be **mechanisms** rather than
requests, because a request is something the model can decline to honour on any given run.

- **Citation** &mdash; not &ldquo;here are the documents that were in context&rdquo;, but *this claim came from
  these characters of that section*.
- **Refusal** &mdash; not &ldquo;the model decided it did not know&rdquo;, but *nothing cleared the floor, so
  there is nothing to answer from*.

## Section 1 &mdash; Bind the claim to the characters

A citation that names a document proves nothing: the document was in the context whatever the
model wrote. A span is checkable.

In [ ]:
def normalise(text: str) -> str:
    return " ".join((text or "").split()).lower()


def find_span(claim: str, chunk: dict):
    """The (start, end) character range in the chunk that supports this claim, or None."""
    hay, needle = normalise(chunk["text"]), normalise(claim)
    i = hay.find(needle)
    # TODO: the range the claim occupies, or None when the chunk does not contain it.
    return BLANK


def bind(claim: str, results: list):
    """Attach the first retrieved chunk that actually contains this claim."""
    for r in results:
        span = find_span(claim, r)
        if span:
            return {"claim": claim, "source": r["source"], "section": r["section"], "span": span}
    return None

In [ ]:
# --- Self-check: Section 1
LIMIT_HITS = search("limit breach approval above 500,000", k=3)
SUPPORTED  = "Payments above USD 500,000 require Treasury approval before release"
INVENTED   = "Payments above USD 500,000 may be released by the duty manager"

check("a supported claim finds its span",
      lambda: bind(SUPPORTED, LIMIT_HITS) is not None)
check("and the span points into the right section",
      lambda: bind(SUPPORTED, LIMIT_HITS)["section"].startswith("3.2"))
check("the span is a real character range",
      lambda: bind(SUPPORTED, LIMIT_HITS)["span"][1] > bind(SUPPORTED, LIMIT_HITS)["span"][0])
check("and it can be checked by slicing the source text",
      lambda: normalise(SUPPORTED) in
              normalise(next(r["text"] for r in LIMIT_HITS
                             if r["section"] == bind(SUPPORTED, LIMIT_HITS)["section"])),
      "an auditor follows one link; the check is a string comparison, not a judgement")
check("AN INVENTED CLAIM BINDS TO NOTHING",
      lambda: bind(INVENTED, LIMIT_HITS) is None,
      "it is plausible, it is about the retrieved topic, and it is not in the text")
check("whitespace differences do not break a real citation",
      lambda: bind("Payments above USD 500,000\n   require Treasury approval", LIMIT_HITS)
              is not None)

## Section 2 &mdash; No span, no claim

The binding is only worth something if an unbound claim is *dropped*. Otherwise you have added a
field, not a control.

In [ ]:
def compose(claims: list, results: list) -> dict:
    """Keep only the claims that can name their source. Report what was dropped."""
    bound = [(c, bind(c, results)) for c in claims]
    # TODO: keep the ones that bound, and record the ones that did not.
    kept = [b for c, b in bound if BLANK]
    dropped = [c for c, b in bound if b is None]
    return {"claims": kept, "dropped": dropped,
            "citations": [f"{b['source']}#{b['section']} [{b['span'][0]}:{b['span'][1]}]"
                          for b in kept]}

In [ ]:
# --- Self-check: Section 2
DRAFT = [SUPPORTED,
         "This does not apply to intra-group transfers",
         INVENTED]

check("the two supported claims survive",
      lambda: len(compose(DRAFT, LIMIT_HITS)["claims"]) == 2)
check("the invented one is dropped",
      lambda: compose(DRAFT, LIMIT_HITS)["dropped"] == [INVENTED])
check("every surviving claim has a citation",
      lambda: len(compose(DRAFT, LIMIT_HITS)["citations"])
              == len(compose(DRAFT, LIMIT_HITS)["claims"]))
check("the citation names a section and a character range",
      lambda: "#3.2" in compose(DRAFT, LIMIT_HITS)["citations"][0]
              and "[" in compose(DRAFT, LIMIT_HITS)["citations"][0])
check("the exception survives alongside the rule, because Lab 6.1 chunked them together",
      lambda: any("intra-group" in b["claim"] for b in compose(DRAFT, LIMIT_HITS)["claims"]),
      "chunk them apart and this claim becomes unciteable, so this control would delete it")
check("dropping is silent to the reader but visible to you",
      lambda: compose(DRAFT, LIMIT_HITS)["dropped"] != [],
      "what got dropped is the most interesting log line in the system")

## Section 3 &mdash; Refuse, usefully

&ldquo;I don't know&rdquo; is a refusal. &ldquo;The runbook covers USD limits and says nothing about FX&rdquo; is
a refusal *and* a work item for whoever owns the corpus.

In [ ]:
FLOOR = 0.25

def respond(question: str, claims=None, floor: float = FLOOR) -> dict:
    """Answer from the corpus, or refuse and say what was missing."""
    results = search(question, k=3, floor=floor)
    if not results:
        nearest = search(question, k=1)          # what we would have used, had we allowed it
        topic = ", ".join(sorted(terms(question))[:4])
        # TODO: refuse. Say what was searched for and what the corpus does have nearby,
        # so the refusal is a work item rather than a shrug.
        return {"answered": False, "citations": [], "why": BLANK}
    out = compose(claims or [], results)
    if not out["claims"]:
        return {"answered": False, "citations": [],
                "why": f"retrieved {len(results)} section(s) but nothing supported a claim"}
    return {"answered": True, "citations": out["citations"],
            "claims": [b["claim"] for b in out["claims"]], "dropped": out["dropped"]}

In [ ]:
# --- Self-check: Section 3
FX_Q = "what is the FX hedging policy for JPY exposure"

check("an answerable question is answered",
      lambda: respond("limit breach approval above 500,000", claims=[SUPPORTED])["answered"]
              is True)
check("and it comes back with a citation",
      lambda: len(respond("limit breach approval above 500,000",
                          claims=[SUPPORTED])["citations"]) == 1)
check("a question the corpus cannot answer is REFUSED",
      lambda: respond(FX_Q, claims=[SUPPORTED])["answered"] is False)
check("the refusal names what was searched for",
      lambda: "hedging" in respond(FX_Q)["why"])
check("and points at the nearest thing the corpus does have",
      lambda: "closest section" in respond(FX_Q)["why"],
      "that sentence is a work item for whoever owns the corpus")
check("retrieving something but supporting nothing also refuses",
      lambda: respond("limit breach approval above 500,000",
                      claims=[INVENTED])["answered"] is False,
      "the second gate: results cleared the floor, and still no claim could name a span")
check("neither refusal asked the model to be careful",
      lambda: respond(FX_Q)["answered"] is False
              and respond("limit breach approval", claims=[INVENTED])["answered"] is False)

def _responses():
    for q, cl in (("limit breach approval above 500,000", [SUPPORTED]),
                  ("limit breach approval above 500,000", [INVENTED]),
                  (FX_Q, [SUPPORTED])):
        r = respond(q, claims=cl)
        print(f"  {'ANSWERED' if r['answered'] else 'REFUSED '}  {q[:40]:42} "
              f"{r.get('citations') or r['why'][:60]}")
guard(_responses)

## Run it for real

Ask the model to answer the FX question from the retrieved context, with and without the floor
applied. This is the experiment that decides whether your grounding is a mechanism or a hope.

In [ ]:
if llm_ready():
    def _grounding():
        for label, floor in (("no floor (all 4 chunks)", 0.0), ("floor 0.25 (nothing)", FLOOR)):
            results = search(FX_Q, k=4, floor=floor)
            context = "\n".join(f"- [{r['section']}] {r['text'][:160]}" for r in results) \
                      or "(no documents were retrieved)"
            reply = ask(f"Context:\n{context}\n\nQuestion: {FX_Q}\n\n"
                        "Answer using ONLY the context. If it does not contain the answer, say so.",
                        system="Be brief.")
            print(f"  [{label}]")
            print(f"      {reply.strip()[:240]}")
            print()
    guard(_grounding)

### Read it

With the floor applied there is no context, so there is nothing to be wrong from &mdash; the refusal
is structural. Without it, four chunks about payment limits are sitting in front of a question
about FX, and the instruction &ldquo;use ONLY the context&rdquo; is the only thing standing between you and
an answer stitched out of the nearest available prose.

Sometimes the model handles it perfectly. That is worth noticing and not worth relying on: you
cannot put &ldquo;the model was sensible&rdquo; in a control document, and it is not the same sentence in
the next model version.

In [ ]:
score()

## Your turn

1. Extractive grounding is the strictest kind and the least fluent. Let the model paraphrase, then
   decide how you would still bind a claim to a span &mdash; and what you lose when the match stops
   being exact.
2. `respond` drops unsupported claims silently. Log them instead, and after a day of traffic read
   the log: the claims a model keeps trying to make and cannot support are a map of what your
   corpus is missing.
3. `FLOOR` is 0.25 because it worked here. Find the value where the FX question refuses and the
   four real questions still answer &mdash; then argue for it with a number rather than a feeling.
   That argument is Lab 6.5.